# Series with SymPy: 

1. **The crystal.** Friday we claimed the bracket $\dfrac{1}{(d+x)^2} - \dfrac{1}{(d-x)^2} \approx -\dfrac{4x}{d^3}$. (For simplicity, let's ignore the leading $\kappa/m$)
2. **A series inside a series.** $\dfrac{1}{1+\sin x}$ through $x^3$, by putting the sine series into $\dfrac{1}{1+a}$.
3. **Taylor about a point.** Approximating $\cos x$ near $x = 3.5$: centered at 0, or centered at 3.


In [ ]:
%matplotlib inline
import numpy as np
from matplotlib import pyplot as plt
import sympy as sp
from ipywidgets import interact

x, a, u = sp.symbols('x a u')

## 1. The crystal

Write the bracket in terms of $u = x/d$, which is small when $x$ is small compared with $d$. Then the bracket is
$\dfrac{1}{d^2}\left[\dfrac{1}{(1+u)^2} - \dfrac{1}{(1-u)^2}\right]$. This factorization is convenient, as the part in square brackets has no $d$ in it.

In [ ]:
bracket = 1/(1 + u)**2 - 1/(1 - u)**2
# every term through u^7; O(u^8) is what is left out
bracket.series(u, 0, 8)        

Well that was easy. We can check here, our linear term is $-4u$; with the $1/d^2$ back in front, that is $-4x/d^3$.

We can ask SymPy to cut it off at the linear or cubic term explicitly (remember that the last argument tells us which order gets removed)

In [ ]:
linear = bracket.series(u, 0, 2).removeO()
cubic = bracket.series(u, 0, 4).removeO()
linear, cubic

Now let's plot the bracket and both approximations. Where does the linear one stop being good?

In [ ]:
us = np.linspace(-0.5, 0.5, 400)
plt.figure()
plt.plot(us, sp.lambdify(u, bracket)(us), 'k--', label='the bracket')
plt.plot(us, sp.lambdify(u, linear)(us), 'red', label=f'linear: ${sp.latex(linear)}$')
plt.plot(us, sp.lambdify(u, cubic)(us), 'purple', label=f'through cubic: ${sp.latex(cubic)}$')
plt.xlabel('u = x/d')
plt.legend(frameon=False)
plt.show()

In [ ]:
# How far off is the linear term? Percent error at a few values of u = x/d
for value in [0.01, 0.05, 0.1, 0.2]:
    exact = bracket.subs(u, value)
    print(f"u = {value}:  linear is off by {float(abs((linear.subs(u, value) - exact) / exact)) * 100:.2f}%")

## 2. A series inside a series: $1/(1 + \sin x)$ through $x^3$

The outer function is $\dfrac{1}{1+a}$; the inner one is $a = \sin x$.

In [ ]:
outer = (1/(1 + a)).series(a, 0, 4).removeO()     # 1 - a + a^2 - a^3
inner = sp.sin(x).series(x, 0, 4).removeO()        # x - x^3/6
outer, inner

Substitute the inner series for $a$ and multiply everything out. When doing this by hand, collecting terms is a pain in the neck. Sympy is quite nice.

In [ ]:
substituted = sp.expand(outer.subs(a, inner))
substituted

Let's keep only the terms through $x^3$, then check against SymPy's own series of $1/(1 + \sin x)$:

In [ ]:
ours = sum(substituted.coeff(x, k) * x**k for k in range(4))
direct = (1/(1 + sp.sin(x))).series(x, 0, 4)
ours, direct

Your turn (Felder 2.92(c)): the Gaussian. Check your whiteboard answer for $\int_0^1 e^{-x^2}\,dx$ from the series through $x^8$
against the computer's value. 

Hint: `sp.exp(-x**2).series(x, 0, 10).removeO()`, then `sp.integrate(..., (x, 0, 1))`;
for the computer's value, `sp.N(sp.integrate(sp.exp(-x**2), (x, 0, 1)))`.

## 3. Taylor about a point

For a Taylor series, we'll want to pick a function, a point about which to expand our series, and the number of terms. 

Here, we'll expand about 3 or 0, and draw a vertical line to indicate which. Then we want to know how good our approximation is at another point. We'll choose that point to be 3.5 and we'll draw a dot there. The printout tells us the error evaluating f(3.5).

`n_terms` counts the powers $0$ through $n-1$, including the ones whose coefficient is zero, so for $\cos x$ and $\sin x$ about 0 every other step adds nothing.

Start with $\cos x$, center 0, and slide the number of terms up. Then switch the center to 3.

In [ ]:
functions = {'cos(x)': sp.cos(x), 'e^x': sp.exp(x), 'sin(x)': sp.sin(x)}
xs = np.linspace(-1, 6, 500)

def values(expression):
    return 0 * xs + sp.lambdify(x, expression)(xs)   # numbers to plot; the 0*xs turns a constant into a flat line

def plot_taylor(function='cos(x)', center=0, n_terms=1, x_eval=3.5):
    f = functions[function]
    # the Taylor formula: f^(k)(a)/k! (x - a)^k
    terms = []
    for k in range(n_terms):
        terms.append(f.diff(x, k).subs(x, center) / sp.factorial(k) * (x - center)**k)
    
    approximation = sum(terms)
    approx = float(approximation.subs(x, x_eval))
    
    exact = float(f.subs(x, x_eval))

    # Now we plot. Slight cuteness to show a few things on the plot.
    plt.figure()
    plt.plot(xs, values(f), 'k--', label=function)
    plt.plot(xs, values(approximation), 'purple', linewidth=3, label=f'{n_terms} term(s) about x = {center}')
    plt.plot([x_eval], [exact], 'ko')
    plt.plot([x_eval], [approx], 'o', color='purple')
    plt.axvline(center, color='grey', linewidth=0.5)
    plt.ylim(-3, 3) if function != 'e^x' else plt.ylim(-5, 150)
    plt.legend(loc='upper left', frameon=False)
    plt.show()
    print(f"at x = {x_eval}: {function} = {exact:.4f}, the approximation gives {approx:.4f}, "
          f"off by {abs(approx - exact) / abs(exact) * 100:.2f}%")

interact(plot_taylor, function=list(functions), 
         center=[0, 3], n_terms=(1, 12), x_eval=(2.0, 5.0, 0.1));

## Exit ticket

Compare approximating the three functions about 0 vs about x=3. Write your answers on a piece of paper and hand in.

*(your answer here if you're doing this virtually)*